Camryn ORS with antibiotic comparison
I tried to use Jake's baseline parameters, equations, and solver so all of our code agrees. My section compares ORS to ORS with antibiotics. The baseline code below is copied from Jake.

In [ ]:
from dataclasses import dataclass, replace
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

Jake baseline

In [1]:
# NEED TO COME BACK:
# r_growth: Jake uses 1.55 /h; sheet says 1609.97/h
# f_planktonic: confirm value
# eta: sheet allows 0 to 1; Jake samples 1 to 1.5
@dataclass(frozen=True)
class Params:
    # --- (1) bacterial load ---
    r_growth: float = 1.55          # 1/h      logistic growth rate              
    N_SI_max: float = 3e13         # cells    carrying capacity                 
    k_imm: float = 0.1             # 1/h      linear immune clearance           TODO placeholder
    f_planktonic: float = 0.2      # -        free-floating fraction [0, 1]     TODO placeholder
    V_lumen: float = 43.0          # mL       SI lumen volume (constant)        

    # --- (2) epithelial activation ---
    k_E_max: float = 0.5           # 1/h      max activation rate (doc: K_max)  TODO placeholder
    K_half_N: float = 1e9          # cells    half-saturation (doc: K_N)        TODO placeholder
    r_turnover: float = 0.05       # 1/h      epithelial turnover               TODO placeholder

    # --- (5), (6) flows ---
    Q_SI_0: float = 62.0           # mL/h     baseline SI outflow               
    Q_SI_max: float = 1050.0       # mL/h     max SI outflow                    
    A_max: float = 200.0           # mL/h     max colonic absorption            TODO placeholder

    # --- (3) whole-body fluid ---
    r_liquid_intake: float = 62.0  # mL/h     drinking/food fluid intake        
    U_liquid: float = 62.0         # mL/h     urine output (= intake to start)  

    # --- (4) whole-body potassium ---
    r_K_intake: float = 3.75        # mmol/h   dietary K+ intake                 
    U_K: float = 3.75               # mmol/h   urinary K+ (= intake to start)    
    c_K_ORS: float = 0.020         # mmol/mL  K+ concentration in ORS           
    c_K_stool: float = 0.015       # mmol/mL  K+ concentration in stool         TODO placeholder

    # --- (7) ORS ---
    eta: float = 1.0               # -        ORS replacement fraction [0, 1]   
    t_R: float = 45.6              # h        ORS start time                  

    # --- initial conditions ---
    N_intake: float = 1e8          # cells    ingested dose                   
    f_cleared: float = 0.99        # -        killed by stomach acid [0, 1]  
    V0: float = 14000.0            # mL       initial whole-body fluid          Updated Eqns doc
    K0: float = 3500.0             # mmol     initial whole-body K+             Updated Eqns doc

    def __post_init__(self):
        for name in ("f_planktonic", "f_cleared"):
            if not 0.0 <= getattr(self, name) <= 1.0:
                raise ValueError(f"{name} must be in [0, 1]")
        if not 0.0 <= self.eta <= 1.5:
            raise ValueError("eta must be in [0, 1.5]")
        for name in ("N_SI_max", "V_lumen", "K_half_N", "V0", "K0"):
            if getattr(self, name) <= 0:
                raise ValueError(f"{name} must be > 0")
        for name in ("r_growth", "k_imm", "k_E_max", "r_turnover", "Q_SI_0", "Q_SI_max",
                     "A_max", "r_liquid_intake", "U_liquid", "r_K_intake", "U_K",
                     "c_K_ORS", "c_K_stool", "t_R", "N_intake"):
            if getattr(self, name) < 0:
                raise ValueError(f"{name} must be >= 0")
        if self.Q_SI_max < self.Q_SI_0:
            raise ValueError("Q_SI_max must be >= Q_SI_0")

    def with_(self, **overrides):
        # Copy with some fields changed (validation runs again).
        return replace(self, **overrides)


p = Params()
p

NameError: name 'dataclass' is not defined

Jake baseline equations

In [2]:
def Q_SI_of(E, p):
    # (5) secretory outflow from the small intestine, mL/h
    return p.Q_SI_0 + (p.Q_SI_max - p.Q_SI_0) * E


def Q_stool_of(Q_SI, p):
    # (6) stool output: what colonic absorption can't take back, mL/h
    return np.maximum(0.0, Q_SI - p.A_max)


def ors(t, Q_stool, p):
    # (7) ORS intake, mL/h. u(t - t_R) is a step: 0 before t_R, 1 from t_R on.
    return p.eta * Q_stool * (np.asarray(t) >= p.t_R)


def rhs(t, y, p):
    N, E, V, K = y

    # Clamp only for evaluating the equations; the solver's own state is untouched.
    N = max(N, 0.0)
    E = min(max(E, 0.0), 1.0)

    Q_SI = Q_SI_of(E, p)
    Q_stool = Q_stool_of(Q_SI, p)
    ORS = ors(t, Q_stool, p)

    dN = (p.r_growth * N * (1.0 - N / p.N_SI_max)
          - p.k_imm * N
          - p.f_planktonic * Q_SI * N / p.V_lumen)                             # (1)
    dE = p.k_E_max * N / (p.K_half_N + N) * (1.0 - E) - p.r_turnover * E     # (2)
    dV = ORS + p.r_liquid_intake - Q_stool - p.U_liquid                      # (3)
    dK = p.r_K_intake + p.c_K_ORS * ORS - p.c_K_stool * Q_stool - p.U_K      # (4)
    return [dN, dE, dV, dK]

My added antibiotic effect
Bacterial clearance term after antibiotics start is k_ab * N. The other equations stay the same, so bacteria must decrease before epithelial and stool output respond.

In [3]:
def rhs_with_antibiotic(t, y, p, k_ab, antibiotic_on):
    # Start with Jake equations with ORS
    dN, dE, dV, dK = rhs(t, y, p)
    # NEED TO COME BACK: k_ab is an effective clearance rate in 1/h
    if antibiotic_on:
        N = max(y[0], 0.0)
        dN = dN - k_ab * N

    return [dN, dE, dV, dK]


I added antibiotic start time as another segment boundary. Each segment has antibiotics either on or off so the solver does not cross that treatment switch in one step.

In [4]:
@dataclass
class Result:
    t: np.ndarray
    N_SI: np.ndarray
    E: np.ndarray
    V: np.ndarray
    K: np.ndarray
    Q_SI: np.ndarray
    Q_stool: np.ndarray
    ORS: np.ndarray
    event_times: list   # times where Q_SI crossed A_max


def initial_state(p):
    return [p.N_intake * (1.0 - p.f_cleared), 0.0, p.V0, p.K0]


def simulate_treatment(p, k_ab=0.0, t_A=None, t_end=72.0, n_points=2000,
                       rtol=1e-8, atol=(1e-3, 1e-10, 1e-6, 1e-6)):
    if not np.isfinite(k_ab) or k_ab < 0:
        raise ValueError("k_ab must be finite and nonnegative")
    if t_A is not None and (not np.isfinite(t_A) or t_A < 0):
        raise ValueError("t_A must be finite and nonnegative")
    if k_ab > 0 and t_A is None:
        raise ValueError("Need t_A when antibiotics are on")
    if not np.isfinite(t_end) or t_end <= 0 or n_points < 2:
        raise ValueError("Need a positive time horizon and at least 2 points")
    t_grid = np.linspace(0.0, t_end, n_points)

    # Restart at ORS start and antibiotic start.
    switches = [p.t_R]
    if t_A is not None:
        switches.append(t_A)
    stops = sorted({t_end, *(t for t in switches if 0 < t < t_end)})

    # A_max crossing only possible if A_max lies strictly between the min and max of Q_SI.
    has_crossing = p.Q_SI_0 < p.A_max < p.Q_SI_max

    def crossing(t, y, p):
        return Q_SI_of(y[1], p) - p.A_max
    crossing.terminal = True

    t_start, y = 0.0, initial_state(p)
    next_dir = None
    ts, ys, event_times = [], [], []

    for t_stop in stops:
        antibiotic_on = t_A is not None and t_start >= t_A and k_ab > 0
        ors_on = t_start >= p.t_R
        # Keep each treatment switch constant throughout this segment.
        p_segment = p.with_(t_R=0.0 if ors_on else t_end + 1.0)

        def segment_rhs(t, y, unused_p):
            return rhs_with_antibiotic(t, y, p_segment, k_ab, antibiotic_on)

        for _ in range(100):  # guard against endless restarts
            events = None
            if has_crossing:
                # Watch for an upward crossing if currently below A_max, else downward.
                crossing.direction = next_dir if next_dir is not None else (1.0 if crossing(t_start, y, p) < 0 else -1.0)
                events = crossing

            mask = (t_grid > t_start) & (t_grid < t_stop)
            t_eval = np.concatenate(([t_start], t_grid[mask], [t_stop]))
            sol = solve_ivp(segment_rhs, (t_start, t_stop), y, args=(p,), t_eval=t_eval,
                            method="LSODA", rtol=rtol, atol=atol, events=events)
            if not sol.success:
                raise RuntimeError(sol.message)

            ts.append(sol.t)
            ys.append(sol.y)

            if sol.status == 1:  # stopped on the A_max event -> restart from it
                t_start = sol.t_events[0][0]
                y = list(sol.y_events[0][0])
                ts.append([t_start])
                ys.append(np.array(y)[:, None])
                event_times.append(t_start)
                next_dir = -crossing.direction
            else:
                t_start, y = t_stop, list(sol.y[:, -1])
                break
        else:
            raise RuntimeError("too many A_max crossings in one segment")

    t = np.concatenate(ts)
    Y = np.concatenate(ys, axis=1)
    t, idx = np.unique(t, return_index=True)  # drop duplicate segment-boundary points
    N, E, V, K = Y[:, idx]

    Q_SI = Q_SI_of(E, p)
    Q_stool = Q_stool_of(Q_SI, p)
    return Result(t, N, E, V, K, Q_SI, Q_stool, ors(t, Q_stool, p), event_times)

NameError: name 'dataclass' is not defined

Choose the treatment switch "At ORS start" means at t_R. All scenarios have the same ORS start time, replacement fraction, initial conditions, and final time.

NEED TO COME BACK: choose k_ab and the delay with the group.50% stool-volume reduction can be a fitting target.

In [5]:
k_ab = None       # NEED TO COME BACK: antibiotic clearance rate, 1/h
delay_h = None    # NEED TO COME BACK: delay after ORS starts, h
t_end = 72.0      # NEED TO COME BACK: confirm common observation window

# Keep Jake baseline
p_compare = p

# NEED TO COME BACK: eta=1 replaces ongoing loss
# NEED TO COME BACK: ORS is unlimited and proportional to stool flow here
# NEED TO COME BACK: no recovery or death stopping criteria are defined

results = {}
if k_ab is None or delay_h is None:
    print("NEED TO COME BACK: enter k_ab and delay_h to run treatment comparisons.")
else:
    if not np.isfinite(delay_h) or delay_h < 0:
        raise ValueError("delay_h must be finite and nonnegative")
    results["ORS alone"] = simulate_treatment(p_compare, t_end=t_end)
    results["ORS + antibiotic at ORS start"] = simulate_treatment(
        p_compare, k_ab=k_ab, t_A=p_compare.t_R, t_end=t_end)
    results["ORS + delayed antibiotic"] = simulate_treatment(
        p_compare, k_ab=k_ab, t_A=p_compare.t_R + delay_h, t_end=t_end)


NameError: name 'p' is not defined

Stool volume and ORS intake. The area under each flow curve is the total volume.

In [6]:
def total_stool_and_ors(r, p):
    # Trapezoid rule written out so this works with older NumPy versions too.
    dt = np.diff(r.t)
    total_stool = np.sum(0.5 * (r.Q_stool[:-1] + r.Q_stool[1:]) * dt)
    # Solver includes the treatment boundary exactly.
    after = r.t >= p.t_R
    t_after = r.t[after]
    q_after = r.Q_stool[after]
    total_ors = p.eta * np.sum(
        0.5 * (q_after[:-1] + q_after[1:]) * np.diff(t_after))
    return total_stool, total_ors

if results:
    reference, _ = total_stool_and_ors(results["ORS alone"], p_compare)
    for label, r in results.items():
        stool, fluid = total_stool_and_ors(r, p_compare)
        reduction = 100 * (1 - stool / reference) if reference > 0 else np.nan
        print(label)
        print(f"  Total stool volume: {stool / 1000:.2f} L")
        print(f"  Reduction versus ORS alone: {reduction:.1f}%")
        print(f"  Total modeled ORS intake: {fluid / 1000:.2f} L")
        print(f"  Minimum fluid volume: {r.V.min():.0f} mL")
        if np.min(r.V) <= 0 or np.min(r.K) <= 0:
            print("  NEED TO COME BACK: nonphysical state; do not interpret this run.")
else:
    print("Results are waiting for treatment settings.")

# NEED TO COME BACK: these totals cover the full 72 h, including before treatment.
# Match the clinical source's timing and population before comparing percentages.
# NEED TO COME BACK: modeled ORS intake excludes extra fluid to correct prior deficit.


NameError: name 'results' is not defined

In [ ]:
if results:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
    for label, r in results.items():
        cumulative_stool = np.r_[0.0, np.cumsum(
            0.5 * (r.Q_stool[:-1] + r.Q_stool[1:]) * np.diff(r.t))]
        axes[0, 0].plot(r.t, r.Q_stool, label=label)
        axes[0, 1].plot(r.t, cumulative_stool / 1000, label=label)
        axes[1, 0].plot(r.t, r.V, label=label)
        axes[1, 1].plot(r.t, r.K, label=label)

    axes[0, 0].set_ylabel("Stool output (mL/h)")
    axes[0, 1].set_ylabel("Cumulative stool volume (L)")
    axes[1, 0].set_ylabel("Fluid compartment volume (mL)")
    axes[1, 1].set_ylabel("Total body potassium (mmol)")
    for ax in axes.flat:
        ax.axvline(p_compare.t_R, color="gray", linestyle=":", linewidth=1)
        ax.set_xlabel("Time since bacteria enter SI (h)")
    axes[0, 0].legend(fontsize=8)
    fig.suptitle("ORS alone and ORS + antibiotic: draft model")
    fig.tight_layout()
    plt.show()
else:
    print("Plots will appear after treatment settings are filled in.")

# NEED TO COME BACK: final figure number, caption, and approved parameters.
# NEED TO COME BACK: add uncertainty analysis if required; these are single runs.


Quick check: With antibiotic clearance set to zero, changing its start time should not change the solution apart from numerical tolerance.

In [ ]:
# This short check can run even while the treatment settings are missing.
a = simulate_treatment(p, k_ab=0.0, t_end=8.0)
b = simulate_treatment(p, k_ab=0.0, t_A=4.0, t_end=8.0)
# Compare V and K after interpolating onto the same times.
assert np.allclose(a.V, np.interp(a.t, b.t, b.V), rtol=1e-5, atol=1e-3)
assert np.allclose(a.K, np.interp(a.t, b.t, b.K), rtol=1e-5, atol=1e-3)
print("Zero-antibiotic fluid and potassium comparison passed.")

# NEED TO COME BACK: Jake's original validation test expects eta=1.5
# to be rejected, although his Params accepts 1.5.


NEED TO COME BACK: Simulated stool output, cumulative stool volume, fluid compartment volume, and total body potassium for ORS alone and ORS plus antibiotics. ORS starts at NEED TO COME BACK h. Antibiotics start at ORS initiation or NEED TO COME BACK h later, with an assumed effective clearance rate of NEED TO COME BACK h⁻¹. The dotted line marks ORS initiation. All scenarios use the same baseline parameters and observation window. NEED TO COME BACK: add the main result after running and resolve the fluid compartment label.